# Shot Reader 8B — video shot to a JSON 3D scene on a free T4

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/donvito/notebooks/blob/main/colab/shot-reader-8b-video-shot-to-3d-scene.ipynb)

[`voyager-so/shot-reader-8b`](https://huggingface.co/voyager-so/shot-reader-8b) is Voyager's **Shot Reader 8B** (Apache-2.0, public): a fine-tune of [`Qwen/Qwen3-VL-8B-Instruct`](https://huggingface.co/Qwen/Qwen3-VL-8B-Instruct) (`Qwen3VLForConditionalGeneration`, Hub ~8.8B, BF16 safetensors **17.5 GB**). It reads **one continuous video shot** and writes a **JSON shot sheet** — camera height, angle, lens, move, who is in frame, where they stand, how they move, and the set — that a 3D tool or a coding agent can rebuild in Blender.

**Why this model.** The [card](https://huggingface.co/voyager-so/shot-reader-8b) argues that scene-building agents can write Blender but do not get the camera right from a reference clip. Shot Reader writes that understanding first. The published recipe is `shot_reader.py` plus `schema.md`: **16 frames sampled evenly, resized to 448×256**, then the card prompt. This notebook uses that **exact prompt and schema**. On a free T4 we load **4-bit NF4 + float16 compute** (`--load-in-4bit` in the script; T4 has **no native bfloat16**) and sample **8 frames** so ~15 GB of VRAM is enough (the card's 16-frame / 8192-token path is for a 16 GB RTX 5080-class card).

See [Use cases](https://github.com/donvito/notebooks/blob/main/docs/shot-reader-8b-video-shot-to-3d-scene.md) for when this notebook is a good fit and when it is not.

**Licence:** Apache-2.0 (same as the Qwen3-VL-8B base). Stock Transformers classes — no `trust_remote_code`, no Hugging Face token.

**Runtime:** *Runtime > Change runtime type > T4 GPU*. A free T4 has ~15–16 GB of VRAM, **no native bfloat16**, and **no FlashAttention-2**. BF16 weights will not fit. The card's 4-bit path is bitsandbytes NF4; `shot_reader.py` uses `bnb_4bit_compute_dtype=torch.bfloat16`, which is the wrong compute type on `sm_75`. This notebook uses **float16 compute**. First run downloads **~17.5 GB**. `config.json` records **`transformers_version: 5.23.0`**; the card asks for `transformers>=4.57`. Colab's preinstalled 5.x may be newer (**5.24+** is current on PyPI). This notebook **pins `transformers==5.23.0`** so a later 5.x processor change cannot silently break Qwen3-VL the way 5.21 broke Mage-VL-Cowork in this repo. **Do not reinstall torch.**

**Not in this notebook:** vLLM (`--backend vllm`), Voyager Cloud / H100 FP8 serving, the unmerged LoRA in `lora/`, or the community [`mradermacher/shot-reader-8b-GGUF`](https://huggingface.co/mradermacher/shot-reader-8b-GGUF) path. Prefer Transformers 4-bit; GGUF is only mentioned as a later llama.cpp option.

**Links:** [model card](https://huggingface.co/voyager-so/shot-reader-8b) · [base Qwen3-VL-8B-Instruct](https://huggingface.co/Qwen/Qwen3-VL-8B-Instruct) · [VibePrevisBench](https://arxiv.org/abs/2611.15789) · [announcement](https://x.com/j_h_h_3/status/2123532630645977367) · [use cases](https://github.com/donvito/notebooks/blob/main/docs/shot-reader-8b-video-shot-to-3d-scene.md) · [Voyager](https://voyager.so)


## 0. Check the GPU

Confirm CUDA is available, print the device name and free VRAM. Need a T4 (or any CUDA GPU). A restart after the install cell drops this state; section 2 prints it again before load.


In [ ]:
import os
import shutil

os.environ["HF_HUB_DISABLE_IMPLICIT_TOKEN"] = "1"

import torch

print("torch.cuda.is_available():", torch.cuda.is_available())
print("torch", torch.__version__, "| cuda", torch.version.cuda)

if torch.cuda.is_available():
    free_b, total_b = torch.cuda.mem_get_info()
    major, minor = torch.cuda.get_device_capability()
    print("GPU name:", torch.cuda.get_device_name(0), f"sm_{major}{minor}")
    print(f"VRAM free / total: {free_b / 1024**3:.1f} / {total_b / 1024**3:.1f} GB")
    if major < 8:
        print("No native bfloat16 on this GPU; 4-bit load will use float16 compute.")
else:
    print("GPU name: none")
    print("VRAM free: n/a")
    print("Select Runtime > Change runtime type > T4 GPU, then rerun from the top.")

print("nvidia-smi:", shutil.which("nvidia-smi") or "not on PATH")
print("ffmpeg:", shutil.which("ffmpeg") or "not on PATH (PyAV fallback will sample frames)")


## 1. Install (keep Colab's torch)

The [card](https://huggingface.co/voyager-so/shot-reader-8b) install is:

```bash
pip install torch "transformers>=4.57" pillow huggingface_hub   # plus ffmpeg on your PATH
```

`--load-in-4bit` also needs **`bitsandbytes`**. `config.json` was saved with **`transformers_version: 5.23.0`**. Architecture is stock `Qwen3VLForConditionalGeneration` / `Qwen3VLProcessor` (no `trust_remote_code`). Qwen3-VL landed in 4.57; **5.23.0** is the export we pin so Colab 5.24.x cannot change the processor. `Qwen3VLProcessor` imports **torchvision** even when we pass still frames. **`av`** samples frames if ffmpeg is missing and writes the synthesized fallback clip.

**Do not reinstall torch** — Colab already ships a CUDA wheel. Use **`%pip`** (not `python -I -m pip`). Isolated mode drops Colab's site-packages and breaks the preinstalled torch. Do **not** install `flash-attn`.

**Do not import the model in this cell.** Colab has already imported its preinstalled transformers. After reconnect, skip this cell and continue from section 2.


In [ ]:
import importlib.metadata as md
import os
import signal
import sys
from pathlib import Path

os.environ["HF_HUB_DISABLE_IMPLICIT_TOKEN"] = "1"

INSTALL_TAG = "tf==5.23.0-shot-reader-8b"
MARKER = (
    Path("/content/.shot_reader_8b_nb_install_ok")
    if Path("/content").is_dir()
    else Path(".shot_reader_8b_nb_install_ok")
)
already_restarted = MARKER.exists() and MARKER.read_text().strip() == INSTALL_TAG

print("python", sys.version.split()[0])
try:
    import torch

    print("torch (keep Colab's wheel)", torch.__version__, "| cuda", torch.version.cuda)
except ImportError:
    print("torch not imported yet")

# Public weights: skip Colab userdata.get("HF_TOKEN") popups.
try:
    import huggingface_hub.utils._auth as _hf_auth

    _hf_auth._IS_GOOGLE_COLAB_CHECKED = True
    _hf_auth._GOOGLE_COLAB_SECRET = None
except Exception:
    pass

# Checkpoint transformers_version: 5.23.0. Card floor is >=4.57.
# Pin exact 5.23.0 so Colab 5.24.x cannot silently break Qwen3-VL.
# Keep Colab's CUDA torch. Do not install flash-attn.
get_ipython().run_line_magic(
    "pip",
    'install -q "transformers==5.23.0" "accelerate>=1.1.0" "bitsandbytes>=0.45.0" torchvision pillow av huggingface_hub',
)

print()
for pkg in (
    "transformers",
    "accelerate",
    "bitsandbytes",
    "huggingface_hub",
    "pillow",
    "av",
):
    try:
        print(pkg, md.version(pkg))
    except md.PackageNotFoundError:
        print(pkg, "not installed")

MARKER.write_text(INSTALL_TAG)
if already_restarted:
    print("Install marker already set; not restarting again. Continue from section 2.")
else:
    print()
    print("=" * 72)
    print("Restarting the session so the pinned transformers is imported.")
    print("After reconnect: skip this cell and run from section 2 (runtime check).")
    print("A 'session crashed' or 'Operation cancelled' message here is the restart,")
    print("not a failed install — pip already finished (versions printed above).")
    print("=" * 72)
    os.kill(os.getpid(), signal.SIGKILL)


> The install cell **restarts the session for you** so the **5.23.0** pin replaces Colab's already-imported transformers. After reconnect, skip it and continue from section 2.
> A "session crashed" message after install is the restart, not a failed pip. If you restarted by hand instead: *Runtime > Restart session*, then this next cell.


## 2. Runtime check after restart

Confirm `transformers==5.23.0`, CUDA, device name, and free VRAM. T4 (`sm_75`) has no native bfloat16 — **4-bit NF4 with float16 compute** is what this notebook loads. Frame count is capped here so a later cell cannot accidentally use the card's 16 frames / 8192 new tokens.


In [ ]:
import gc
import os
import shutil

os.environ["HF_HUB_DISABLE_IMPLICIT_TOKEN"] = "1"
try:
    import huggingface_hub.utils._auth as _hf_auth

    _hf_auth._IS_GOOGLE_COLAB_CHECKED = True
    _hf_auth._GOOGLE_COLAB_SECRET = None
except Exception:
    pass

import torch
import transformers
from packaging.version import Version

loaded = Version(transformers.__version__.split("+")[0])
assert loaded == Version("5.23.0"), (
    f"transformers {transformers.__version__} is loaded; Shot Reader 8B pins "
    "exactly 5.23.0 (config.json transformers_version). "
    "Run the install cell, then Runtime > Restart session."
)

print("torch.cuda.is_available():", torch.cuda.is_available())
if not torch.cuda.is_available():
    raise SystemExit("Select Runtime > Change runtime type > T4 GPU, then rerun from the top.")

# T4 (sm_75) has no FlashAttention-2. Keep PyTorch SDPA on mem-efficient and math kernels.
torch.backends.cuda.enable_flash_sdp(False)
torch.backends.cuda.enable_mem_efficient_sdp(True)
torch.backends.cuda.enable_math_sdp(True)

major, minor = torch.cuda.get_device_capability()
try:
    NATIVE_BF16 = torch.cuda.is_bf16_supported(including_emulation=False)
except TypeError:
    NATIVE_BF16 = major >= 8

DEVICE = "cuda"
# Card 4-bit path uses bfloat16 compute. T4 has no native bf16 — float16 compute.
COMPUTE_DTYPE = torch.float16
MODEL_ID = "voyager-so/shot-reader-8b"
# Card: 16 frames at 448x256, max_new_tokens=8192. T4 smoke: 8 frames, 2048 new tokens.
N_FRAMES = 8
FRAME_SIZE = (448, 256)  # (width, height) — shot_reader.py SIZE
MAX_SHOT_S = 5.0
MAX_NEW_TOKENS = 2048

free_b, total_b = torch.cuda.mem_get_info()
print("transformers", transformers.__version__)
print("torch", torch.__version__)
print("GPU name:", torch.cuda.get_device_name(0), f"sm_{major}{minor}")
print(f"VRAM free / total: {free_b / 1024**3:.1f} / {total_b / 1024**3:.1f} GB")
print("native bf16:", NATIVE_BF16, "| FlashAttention-2: off (sdpa / eager only)")
print("load path: 4-bit NF4 + float16 compute (17.5 GB bf16 does not fit a T4)")
print(
    f"frames: {N_FRAMES} at {FRAME_SIZE[0]}x{FRAME_SIZE[1]} | "
    f"max shot {MAX_SHOT_S:.0f}s | max_new_tokens {MAX_NEW_TOKENS}"
)
print("ffmpeg:", shutil.which("ffmpeg") or "not on PATH")
if major < 8:
    print("No native bfloat16 on this GPU — staying on float16 compute.")


def print_vram(label=""):
    suffix = f"  ({label})" if label else ""
    allocated = torch.cuda.memory_allocated() / 1024**3
    reserved = torch.cuda.memory_reserved() / 1024**3
    peak = torch.cuda.max_memory_allocated() / 1024**3
    free_now, total_now = torch.cuda.mem_get_info()
    print(
        f"VRAM allocated: {allocated:.2f} GB | reserved: {reserved:.2f} GB | "
        f"peak: {peak:.2f} GB | free: {free_now / 1024**3:.1f}/{total_now / 1024**3:.1f} GB{suffix}"
    )


## 3. Load Shot Reader 8B in 4-bit NF4

Stock `AutoProcessor` + `AutoModelForImageTextToText` (`Qwen3VLForConditionalGeneration` / `Qwen3VLProcessor`), matching [`shot_reader.py`](https://huggingface.co/voyager-so/shot-reader-8b/blob/main/shot_reader.py). No `trust_remote_code`. The first run downloads the **17.5 GB** BF16 file, then packs the weights to 4-bit.

The script's 4-bit config is NF4 with **bfloat16** compute. This cell only changes **`bnb_4bit_compute_dtype=torch.float16`** on T4 and tries **SDPA**, then **eager**. `shot_reader.py` comments ~6 GB of VRAM for `--load-in-4bit`. Expected peak here is **~6–8 GB** with 8 frames and 2048 new tokens — an estimate from the packed weights plus vision/KV, not a measured Colab number yet.


In [ ]:
from transformers import AutoModelForImageTextToText, AutoProcessor, BitsAndBytesConfig

QUANT = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=COMPUTE_DTYPE,
    bnb_4bit_use_double_quant=True,
)

torch.cuda.empty_cache()
torch.cuda.reset_peak_memory_stats()

processor = AutoProcessor.from_pretrained(MODEL_ID, token=False)

ATTN = None
model = None
last_error = None


def load_vlm(attn):
    kwargs = dict(
        quantization_config=QUANT,
        device_map="auto",
        low_cpu_mem_usage=True,
        attn_implementation=attn,
        token=False,
    )
    return AutoModelForImageTextToText.from_pretrained(MODEL_ID, **kwargs).eval()


for attn in ("sdpa", "eager"):
    try:
        print(f"loading 4-bit NF4 + {attn}…")
        model = load_vlm(attn)
        ATTN = attn
        break
    except Exception as exc:
        last_error = exc
        print(f"{attn} 4-bit load failed:", type(exc).__name__, exc)
        try:
            del model
        except NameError:
            pass
        model = None
        gc.collect()
        torch.cuda.empty_cache()

if model is None:
    raise RuntimeError("could not load voyager-so/shot-reader-8b in 4-bit") from last_error

if getattr(model, "generation_config", None) is not None:
    model.generation_config.do_sample = False
    model.generation_config.use_cache = True

DEVICE = next(model.parameters()).device
n_params = sum(p.numel() for p in model.parameters()) / 1e9
print("loaded", MODEL_ID)
print("class:", type(model).__name__)
print("processor:", type(processor).__name__)
print("attention:", ATTN, "| quantization: 4-bit nf4 / float16 compute")
print(f"parameters: {n_params:.2f}B  (Hub: ~8.8B)")
print("device:", DEVICE)
print_vram("after load")


## 4. Card prompt, schema, and frame sampler

The model sees **evenly sampled frames resized to 448×256**, then the prompt in `shot_reader.py` (`PROMPT` plus `schema.md`). The text below is copied from that file. `schema.md` is downloaded from the Hub so it stays the card's field reference.

One input is **one continuous shot**. Demos cap each clip at **5 s**. Name subjects as `"ID: class (optional description)"`; the model keeps those ids. If you pass none, the prompt tells it to name people `P01, P02, …` and objects by class.

**Colab cuDNN workaround.** Some Colab T4 images fail Qwen3-VL vision patch embedding during generate with `CUDNN_STATUS_SUBLIBRARY_VERSION_MISMATCH`. The helper disables cuDNN around `generate`, matching the NetraV4 notebook.


In [ ]:
import json
import math
import re
import shutil
import subprocess
import tempfile
import time
from pathlib import Path

from huggingface_hub import hf_hub_download
from IPython.display import display
from PIL import Image

# Exact PROMPT from voyager-so/shot-reader-8b shot_reader.py.
PROMPT = """You are a previz shot reader. You see {nf} frames sampled evenly from one shot of {dur:.2f} s
(frame i is at t = {times}).

Write the shot's fact sheet: one JSON object exactly in the schema below. Measure positions in metres
in the ground frame defined below (origin under the camera in the FIRST frame, +x right, +y forward).
Use object sizes (people are about 1.7 m tall, doors 2 m, chairs 0.9 m, etc.) and perspective to
estimate distances. Give every listed subject a path with at least a key at t=0 and t={dur:.2f}
(add intermediate keys when it moves). Keep the given ids.

Subjects in the brief (id: class): {subjects}

{schema}

Reply with only the JSON object, no prose, no code fences."""

SCHEMA_PATH = hf_hub_download(MODEL_ID, "schema.md", token=False)
SCHEMA = Path(SCHEMA_PATH).read_text()
print("schema.md", SCHEMA_PATH, f"({len(SCHEMA)} chars)")


def video_duration(path):
    path = str(path)
    if shutil.which("ffprobe"):
        out = subprocess.run(
            [
                "ffprobe",
                "-v",
                "error",
                "-show_entries",
                "format=duration",
                "-of",
                "csv=p=0",
                path,
            ],
            capture_output=True,
            text=True,
            check=True,
        ).stdout.strip()
        return float(out)
    import av

    with av.open(path) as container:
        if container.duration is not None and container.duration > 0:
            return container.duration / 1_000_000
        stream = container.streams.video[0]
        if stream.duration and stream.time_base:
            return float(stream.duration * stream.time_base)
    raise RuntimeError(f"could not read duration of {path}")


def sample_frames(path, dur, n_frames=N_FRAMES, size=FRAME_SIZE):
    """Card sampler: n frames at t = i * dur / n, resized to 448x256 BICUBIC."""
    times = [i * dur / n_frames for i in range(n_frames)]
    imgs = []
    if shutil.which("ffmpeg"):
        with tempfile.TemporaryDirectory() as tmp:
            for i, t in enumerate(times):
                f = Path(tmp) / f"{i:02d}.jpg"
                subprocess.run(
                    [
                        "ffmpeg",
                        "-v",
                        "error",
                        "-y",
                        "-ss",
                        f"{t:.3f}",
                        "-i",
                        str(path),
                        "-frames:v",
                        "1",
                        "-q:v",
                        "2",
                        str(f),
                    ],
                    check=True,
                )
                imgs.append(Image.open(f).convert("RGB").resize(size, Image.BICUBIC))
        return times, imgs

    import av

    wanted = list(times)
    picked = {}
    with av.open(str(path)) as container:
        stream = container.streams.video[0]
        for decoded in container.decode(stream):
            t = float(decoded.time or 0.0)
            for i, target in enumerate(wanted):
                if i in picked:
                    continue
                if t + 1e-3 >= target or abs(t - target) < 0.05:
                    picked[i] = decoded.to_image().convert("RGB").resize(size, Image.BICUBIC)
                    break
            if len(picked) == n_frames:
                break
    if len(picked) != n_frames:
        raise RuntimeError(f"PyAV decoded {len(picked)}/{n_frames} frames from {path}")
    imgs = [picked[i] for i in range(n_frames)]
    return times, imgs


def build_prompt(dur, times, subjects, schema=SCHEMA, n_frames=N_FRAMES):
    subs = (
        ", ".join(subjects)
        if subjects
        else "none given (name people P01, P02, ... and objects by class)"
    )
    return PROMPT.format(
        nf=n_frames,
        dur=dur,
        times=", ".join(f"{t:.2f}" for t in times),
        subjects=subs,
        schema=schema,
    )


def chat_text(prompt_text, n_frames=N_FRAMES):
    msgs = [
        {
            "role": "user",
            "content": [{"type": "image"}] * n_frames + [{"type": "text", "text": prompt_text}],
        }
    ]
    return processor.apply_chat_template(msgs, add_generation_prompt=True, tokenize=False)


def parse_sheet(raw):
    match = re.search(r"\{.*\}", raw, flags=re.S)
    if not match:
        return None, raw
    try:
        return json.loads(match.group(0)), raw
    except json.JSONDecodeError:
        return None, raw


def show_contact_sheet(imgs, times, cols=4, thumb=160):
    rows = math.ceil(len(imgs) / cols)
    sheet = Image.new("RGB", (cols * thumb, rows * thumb), (20, 20, 20))
    for i, im in enumerate(imgs):
        r, c = divmod(i, cols)
        sheet.paste(im.resize((thumb, thumb), Image.BICUBIC), (c * thumb, r * thumb))
    display(sheet)
    print("frame times (s):", ", ".join(f"{t:.2f}" for t in times))


def read_shot(path, subjects=None, max_shot_s=MAX_SHOT_S, max_new_tokens=MAX_NEW_TOKENS):
    subjects = list(subjects or [])
    full_dur = video_duration(path)
    dur = min(float(full_dur), float(max_shot_s))
    print(f"{Path(path).name}: {full_dur:.2f}s on disk, using first {dur:.2f}s as one shot")
    times, imgs = sample_frames(path, dur)
    show_contact_sheet(imgs, times)
    prompt_text = build_prompt(dur, times, subjects)
    templated = chat_text(prompt_text)
    enc = processor(text=[templated], images=imgs, return_tensors="pt")
    enc = enc.to(device=DEVICE)
    for key, value in list(enc.items()):
        if torch.is_tensor(value) and torch.is_floating_point(value):
            enc[key] = value.to(dtype=COMPUTE_DTYPE)

    prompt_len = enc["input_ids"].shape[1]
    print(f"prompt tokens: {prompt_len} | max_new_tokens: {max_new_tokens} | subjects: {subjects or '(none given)'}")

    # Colab T4 cuDNN version-mismatch workaround (Qwen3-VL vision patch embedding).
    torch.backends.cudnn.enabled = False
    torch.backends.cudnn.benchmark = False
    t0 = time.perf_counter()
    with torch.inference_mode():
        out = model.generate(
            **enc,
            max_new_tokens=max_new_tokens,
            do_sample=False,
        )
    elapsed = time.perf_counter() - t0
    raw = processor.tokenizer.decode(out[0, prompt_len:], skip_special_tokens=True)
    print(f"generate {elapsed:.1f}s | new tokens: {out.shape[-1] - prompt_len}")
    print_vram("after generate")
    sheet, raw = parse_sheet(raw)
    if sheet is None:
        print("could not parse a JSON sheet; raw output follows")
        print(raw)
        return None
    print(json.dumps(sheet, indent=2))
    return sheet


## 5. Sample clips

Two short **public** clips (people walking, then a tennis rally) so the shot sheet has camera + people to describe. Downloads use a descriptive User-Agent. If a URL fails, the next one is tried; last resort is a synthesized interior with a moving “person” block so later cells still run.


In [ ]:
import urllib.request

import av
import numpy as np
from IPython.display import Video

SAMPLE_DIR = Path("shot_reader_8b_samples")
SAMPLE_DIR.mkdir(exist_ok=True)
UA = {
    "User-Agent": (
        "donvito-notebooks/1.0 (educational Colab demo; "
        "https://github.com/donvito/notebooks)"
    )
}

PEOPLE_URLS = [
    (
        "https://github.com/intel-iot-devkit/sample-videos/raw/master/people-detection.mp4",
        "people-detection.mp4",
    ),
    (
        "https://raw.githubusercontent.com/Tencent/WeMM-Video/main/examples/media/apollo-11-first-steps.mp4",
        "apollo-11-first-steps.mp4",
    ),
]
TENNIS_URLS = [
    (
        "https://huggingface.co/datasets/hf-internal-testing/fixtures_videos/resolve/main/tennis.mp4",
        "tennis.mp4",
    ),
    (
        "https://huggingface.co/datasets/openbmb/DemoCase/resolve/main/football.mp4",
        "football.mp4",
    ),
]


def _urlretrieve(url, path, timeout=120):
    req = urllib.request.Request(url, headers=UA)
    with urllib.request.urlopen(req, timeout=timeout) as resp, open(path, "wb") as handle:
        handle.write(resp.read())
    return Path(path)


def download_first(urls_and_names):
    last_err = None
    for url, name in urls_and_names:
        dest = SAMPLE_DIR / name
        try:
            if not dest.exists() or dest.stat().st_size < 32:
                print("Downloading", url)
                _urlretrieve(url, dest)
            if dest.stat().st_size < 32:
                raise OSError(f"{dest} is too small")
            print(f"loaded {dest.name}: {dest.stat().st_size / 1024:.1f} KiB")
            return dest
        except Exception as exc:
            last_err = exc
            print(f"failed {url}: {exc!r}")
            dest.unlink(missing_ok=True)
    raise RuntimeError(f"all downloads failed: {last_err!r}")


def write_fallback_clip(path, n_frames=20, fps=4, width=448, height=256):
    # Interior-like set: floor + back wall + a person-sized block that walks right.
    container = av.open(str(path), mode="w")
    stream = container.add_stream("mpeg4", rate=fps)
    stream.width = width
    stream.height = height
    stream.pix_fmt = "yuv420p"
    for i in range(n_frames):
        frame = np.zeros((height, width, 3), dtype=np.uint8)
        frame[: 2 * height // 3] = (40, 70, 90)  # wall
        frame[2 * height // 3 :] = (90, 80, 60)  # floor
        x = int(40 + (i / max(n_frames - 1, 1)) * (width - 120))
        y = height // 2
        frame[y : y + 80, x : x + 36] = (210, 160, 120)  # person
        frame[y + 20 : y + 70, 280:360] = (70, 50, 40)  # crate
        vf = av.VideoFrame.from_ndarray(frame, format="rgb24")
        for packet in stream.encode(vf):
            container.mux(packet)
    for packet in stream.encode():
        container.mux(packet)
    container.close()
    return Path(path)


def load_clip(label, urls):
    try:
        path = download_first(urls)
        print(f"{label}:", path)
        return path
    except Exception as exc:
        print(f"{label} download failed:", repr(exc))
        path = write_fallback_clip(SAMPLE_DIR / f"fallback-{label}.mp4")
        print("wrote", path)
        return path


PEOPLE_PATH = load_clip("people", PEOPLE_URLS)
TENNIS_PATH = load_clip("action", TENNIS_URLS)
print("people clip:", PEOPLE_PATH)
print("action clip:", TENNIS_PATH)


## 6. Demo 1 — people in frame (named subjects)

Pedestrian clip (Intel sample video; Apollo 11 / synthesized fallback). Subjects are named the way `shot_reader.py --subject` expects, so the sheet keeps those ids. Pretty-print the JSON shot sheet.


In [ ]:
display(Video(str(PEOPLE_PATH), embed=True, width=448))

DEMO1_SUBJECTS = [
    "P01: person",
    "P02: person",
]
if PEOPLE_PATH.name.startswith("fallback"):
    DEMO1_SUBJECTS = ["P01: person", "crate_1: crate"]

sheet1 = read_shot(PEOPLE_PATH, subjects=DEMO1_SUBJECTS)


## 7. Demo 2 — camera follows action

Tennis rally (Hugging Face fixture; football / synthesized fallback). No `--subject` ids this time: the prompt's fallback line asks the model to name people `P01, P02, …` and objects by class.


In [ ]:
display(Video(str(TENNIS_PATH), embed=True, width=448))

sheet2 = read_shot(TENNIS_PATH, subjects=None)

print()
print(f"peak VRAM this session: {torch.cuda.max_memory_allocated() / 1024**3:.2f} GB")
print_vram("end of demos")


## 8. Caveats and cleanup

**What this notebook is for.** The card's Transformers path on a free T4: sample frames the way `shot_reader.py` does, send the **exact card prompt + `schema.md`**, and pretty-print the JSON shot sheet on two short public clips.

**What it is not.** vLLM, Voyager Cloud H100 FP8 (`~0.002` / 5.9 s per shot on the card are those numbers, not this T4 run), the unmerged LoRA, or GGUF. Community GGUF: [`mradermacher/shot-reader-8b-GGUF`](https://huggingface.co/mradermacher/shot-reader-8b-GGUF).

**Card limitations (quoted, not re-measured).** Events and routes are its weakest fields on real footage; positions are estimates from a single camera, typically within a few metres; input is one continuous shot at a time. Cut longer videos at the edits first.

**T4 / VRAM.** 4-bit NF4 + float16 compute + SDPA. **8 frames** at 448×256 and **2048** new tokens (card: 16 frames / 8192). `shot_reader.py` comments ~6 GB for 4-bit; expected peak here **~6–8 GB**. The number below is `torch.cuda.max_memory_allocated` for this session. Do not load the 17.5 GB BF16 file in float16. Do not install FlashAttention-2. cuDNN is disabled around `generate` because Qwen3-VL `conv3d` has mismatched cuDNN on some Colab T4 images.

**Transformers pin.** `config.json` records **5.23.0**. The card floor is `>=4.57`. This notebook pins **5.23.0** and restarts so Colab 5.24.x is not used.

**Related notebooks:** [WeMM-Video-2B](WeMM-Video-2B-audio-visual-video-understanding.ipynb) (audio-visual describe), [LFM3-VL-2B](LFM3-VL-2B-vision.ipynb), [MiniCPM-V 4.7 1B](MiniCPM-V-4.7-1B-vision.ipynb), [LFM3-Omni-2B](LFM3-Omni-2B.ipynb).

Free the GPU before you open another notebook:


In [ ]:
print_vram("before cleanup")
print(f"peak VRAM this session: {torch.cuda.max_memory_allocated() / 1024**3:.2f} GB")
try:
    del model
except NameError:
    pass
try:
    del processor
except NameError:
    pass
gc.collect()
torch.cuda.empty_cache()
print_vram("after cleanup")
print("Shot Reader 8B released. Re-run the load cell if you want another clip.")
